In [5]:
from pathlib import Path
import pandas as pd
import numpy as np


ACTIVITY125_FILES = [
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Yifeng Mao/data/active_lives_1516_london_125.csv",
        "year": 1,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Yifeng Mao/data/active_lives_1617_london_125.csv",
        "year": 2,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Siyan Xin/2017~2018/2017_data_125_activities.csv",
        "year": 3,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Siyan Xin/2018~2019/2018_data_125_activities.csv",
        "year": 4,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/code/1920_london32_stable125.csv",
        "year": 5,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/code/2021_london32_stable125.csv",
        "year": 6,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Jingyi Hua/data/processed/year7_125activities.csv",
        "year": 7,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Jingyi Hua/data/processed/year8_125activities.csv",
        "year": 8,
    },
]


OUTPUT_DIR = Path("/Users/zsh/Downloads/PROJECT/code")

WEIGHT_COL = "wt_final"

MISSING_CODES = [-99, -98, -97, -96, -95, -94, -93, -92, -91]

In [6]:
AGE9_LABELS = {
    2: "16-24",
    3: "25-34",
    4: "35-44",
    5: "45-54",
    6: "55-64",
    7: "65-74",
    8: "75-84",
    9: "85+",
}

MONTHS12_PREFIX = "MONTHS_12_"
DAYS10P60GR_PREFIX = "DAYS10P60GR_"

In [7]:
def weighted_rate(data, value_col, positive_values, weight_col):
    values = data[value_col]
    weights = data[weight_col]

    mask = values.notna() & weights.notna()

    if mask.sum() == 0:
        return np.nan

    if weights.loc[mask].sum() == 0:
        return np.nan

    indicator = values.loc[mask].isin(positive_values).astype(float)

    return np.average(
        indicator,
        weights=weights.loc[mask]
    )

In [8]:
def get_activity_suffixes_from_files(file_info_list):
    suffixes = set()

    for file_info in file_info_list:
        file_path = Path(file_info["path"])
        columns = pd.read_csv(file_path, nrows=0).columns

        for col in columns:
            if col.startswith(MONTHS12_PREFIX):
                suffixes.add(col[len(MONTHS12_PREFIX):])

            if col.startswith(DAYS10P60GR_PREFIX):
                suffixes.add(col[len(DAYS10P60GR_PREFIX):])

    suffixes = sorted(suffixes)

    print(f"Total activity suffixes found: {len(suffixes)}")

    return suffixes

activity_suffixes = get_activity_suffixes_from_files(
    ACTIVITY125_FILES
)

Total activity suffixes found: 125


In [9]:
def standardise_base_columns(df):
    df = df.copy()

    rename_dict = {}

    if "age16plus" in df.columns and "Age16plus" not in df.columns:
        rename_dict["age16plus"] = "Age16plus"

    if "Month" in df.columns and "month" not in df.columns:
        rename_dict["Month"] = "month"

    df = df.rename(columns=rename_dict)

    return df

In [10]:
def make_one_year_activity_panel(
    file_path,
    year_value,
    activity_suffixes,
    weight_col=WEIGHT_COL
):
    file_path = Path(file_path)

    df = pd.read_csv(file_path)
    df = standardise_base_columns(df)

    df = df.replace(MISSING_CODES, np.nan)

    df["year"] = year_value
    df["age_group"] = df["Age9"].map(AGE9_LABELS)

    df = df[
        (df["Age16plus"] == 1)
        & df["age_group"].notna()
    ].copy()

    rows = []

    group_cols = [
        "year",
        "LA_2023",
        "age_group",
    ]

    for group_values, group_data in df.groupby(group_cols, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        base_row = dict(zip(group_cols, group_values))

        for suffix in activity_suffixes:
            months12_col = MONTHS12_PREFIX + suffix
            days10p60gr_col = DAYS10P60GR_PREFIX + suffix

            row = base_row.copy()
            row["activity_suffix"] = suffix

            row["months12_available"] = months12_col in df.columns
            row["days10p60gr_available"] = days10p60gr_col in df.columns

            if row["months12_available"]:
                row["months12_rate"] = weighted_rate(
                    data=group_data,
                    value_col=months12_col,
                    positive_values=[1],
                    weight_col=weight_col
                )

                row["n_months12"] = (
                    group_data[months12_col].notna()
                    & group_data[weight_col].notna()
                ).sum()

                row["weighted_n_months12"] = group_data.loc[
                    group_data[months12_col].notna()
                    & group_data[weight_col].notna(),
                    weight_col
                ].sum()

            else:
                row["months12_rate"] = np.nan
                row["n_months12"] = 0
                row["weighted_n_months12"] = np.nan

            if row["days10p60gr_available"]:
                row["days10p60gr_rate"] = weighted_rate(
                    data=group_data,
                    value_col=days10p60gr_col,
                    positive_values=[1, 2, 3, 4, 5],
                    weight_col=weight_col
                )

                row["n_days10p60gr"] = (
                    group_data[days10p60gr_col].notna()
                    & group_data[weight_col].notna()
                ).sum()

                row["weighted_n_days10p60gr"] = group_data.loc[
                    group_data[days10p60gr_col].notna()
                    & group_data[weight_col].notna(),
                    weight_col
                ].sum()

            else:
                row["days10p60gr_rate"] = np.nan
                row["n_days10p60gr"] = 0
                row["weighted_n_days10p60gr"] = np.nan

            row["small_cell_months12"] = row["n_months12"] < 30
            row["small_cell_days10p60gr"] = row["n_days10p60gr"] < 30

            rows.append(row)

    result = pd.DataFrame(rows)

    print(
        f"Year {year_value}: {len(result)} rows, "
        f"{result['activity_suffix'].nunique()} activities"
    )

    return result

In [11]:
def make_activity_participation_panel(
    file_info_list,
    activity_suffixes,
    output_path,
    weight_col=WEIGHT_COL
):
    panels = []

    for file_info in file_info_list:
        one_year_panel = make_one_year_activity_panel(
            file_path=file_info["path"],
            year_value=file_info["year"],
            activity_suffixes=activity_suffixes,
            weight_col=weight_col
        )

        panels.append(one_year_panel)

    panel_df = pd.concat(
        panels,
        ignore_index=True
    )

    panel_df = panel_df.sort_values(
        [
            "year",
            "LA_2023",
            "age_group",
            "activity_suffix",
        ]
    ).reset_index(drop=True)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    panel_df.to_csv(output_path, index=False)

    print(f"Done: {len(panel_df)} rows")
    print("Years:")
    print(panel_df["year"].value_counts().sort_index())

    print("Activities:")
    print(panel_df["activity_suffix"].nunique())

    return panel_df

activity_participation_panel_full = make_activity_participation_panel(
    file_info_list=ACTIVITY125_FILES,
    activity_suffixes=activity_suffixes,
    output_path=OUTPUT_DIR / "q3_activity_participation_panel_full.csv",
    weight_col=WEIGHT_COL
)

Year 1: 32000 rows, 125 activities
Year 2: 32000 rows, 125 activities
Year 3: 32000 rows, 125 activities
Year 4: 32000 rows, 125 activities
Year 5: 32000 rows, 125 activities
Year 6: 32000 rows, 125 activities
Year 7: 32000 rows, 125 activities
Year 8: 32000 rows, 125 activities
Done: 256000 rows
Years:
year
1    32000
2    32000
3    32000
4    32000
5    32000
6    32000
7    32000
8    32000
Name: count, dtype: int64
Activities:
125


In [12]:
def make_complete_activity_panel(panel_df, output_path):
    availability = (
        panel_df
        .groupby("activity_suffix")
        .agg(
            years_total=("year", "nunique"),
            months12_all_available=("months12_available", "all"),
            days10p60gr_all_available=("days10p60gr_available", "all"),
        )
        .reset_index()
    )

    complete_activities = availability[
        (availability["years_total"] == 8)
        & (availability["months12_all_available"])
        & (availability["days10p60gr_all_available"])
    ]["activity_suffix"].tolist()

    complete_df = panel_df[
        panel_df["activity_suffix"].isin(complete_activities)
    ].copy()

    complete_df.to_csv(output_path, index=False)

    print(f"Complete activities: {len(complete_activities)}")
    print(f"Rows in complete panel: {len(complete_df)}")

    return complete_df, availability

activity_participation_panel_complete, activity_availability = make_complete_activity_panel(
    panel_df=activity_participation_panel_full,
    output_path=OUTPUT_DIR / "q3_activity_participation_panel_complete.csv"
)

activity_availability.to_csv(
    OUTPUT_DIR / "q3_activity_availability_summary.csv",
    index=False
)

Complete activities: 124
Rows in complete panel: 253952


In [13]:
def make_top_activity_dataset(panel_df, output_path):
    rows = []

    group_cols = [
        "year",
        "LA_2023",
        "age_group",
    ]

    for group_values, group_data in panel_df.groupby(group_cols, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        row = dict(zip(group_cols, group_values))

        months12_valid = group_data[
            group_data["months12_rate"].notna()
        ].copy()

        days_valid = group_data[
            group_data["days10p60gr_rate"].notna()
        ].copy()

        if len(months12_valid) > 0:
            top_months12 = months12_valid.sort_values(
                "months12_rate",
                ascending=False
            ).iloc[0]

            row["top_months12_activity"] = top_months12["activity_suffix"]
            row["top_months12_rate"] = top_months12["months12_rate"]
        else:
            row["top_months12_activity"] = np.nan
            row["top_months12_rate"] = np.nan

        if len(days_valid) > 0:
            top_days = days_valid.sort_values(
                "days10p60gr_rate",
                ascending=False
            ).iloc[0]

            row["top_days10p60gr_activity"] = top_days["activity_suffix"]
            row["top_days10p60gr_rate"] = top_days["days10p60gr_rate"]
        else:
            row["top_days10p60gr_activity"] = np.nan
            row["top_days10p60gr_rate"] = np.nan

        rows.append(row)

    result = pd.DataFrame(rows)

    result.to_csv(output_path, index=False)

    print(f"Done: {len(result)} rows")

    return result

top_activity_history = make_top_activity_dataset(
    panel_df=activity_participation_panel_complete,
    output_path=OUTPUT_DIR / "q3_top_activity_history.csv"
)

Done: 2048 rows
